# 💻 Notebook do Aluno — Aula 07: RAG avançado — chunking estratégico, reranking e RAGAS

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 07/14 — Módulo 2: RAG · 🏁 Entrega CKP02**  
**⏱️ 1h40min**  
**📊 faithfulness · answer_relevancy**  
**🏁 CKP02 entrega**  

---

## 🎯 Objetivo da aula

Sair da fase "funciona" para a fase "funciona bem". Medir objetivamente a qualidade do RAG com RAGAS, experimentar duas estratégias de chunking e documentar qual configuração entrega melhores resultados para o domínio do grupo. Isso é o CKP02.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

## 📋 Roteiro do Lab

**Lab — Aula 07 · CKP02 R2 e R3**  
### Otimizar RAG com RAGAS — 2 estratégias de chunking ★★★

*Grupo 3–4 · 20 minutos · Google Colab · PDFs do domínio obrigatórios*

1. Complete as 4 lacunas — carregar PDFs, instanciar SemanticChunker, montar retriever_b e preencher PERGUNTAS com 5 perguntas reais do domínio.
2. Compare as duas estratégias — documente em uma célula markdown qual obteve melhores scores de faithfulness e answer_relevancy e por que (hipótese do grupo).
3. Análise de falhas: identifique 1 pergunta onde a chain errou (faithfulness baixo) — recupere os chunks que foram usados e explique por que o retriever não trouxe o trecho certo.

> **🎯 Gabarito das lacunas**
>
> Lacuna 1: PyMuPDFLoader(pdf).load()
>
> Lacuna 2: SemanticChunker(embeddings)
>
> Lacuna 3: db_b.as_retriever(search_kwargs={"k": 3})
>
> Lacuna 4: PERGUNTAS (no loop de contexts)

---

## 🧩 Notebook Aluno — 55% de lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-community langchain-ollama pymupdf chromadb ragas datasets langchain-experimental langchain-text-splitters -q

# Setup (idêntico à Aula 06)
import os
from google.colab import userdata, files
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# 👉 LACUNA 1: carregue os PDFs do grupo (mesmos da Aula 06)
paginas = []
for pdf in pdf_paths:
    paginas.extend(PyMuPDFLoader(___).load())

# ── ESTRATÉGIA A: Recursive com chunk_size=512 (vs. 800 da Aula 06) ──
chunks_a = RecursiveCharacterTextSplitter(
    chunk_size=___, chunk_overlap=___
).split_documents(paginas)
db_a         = Chroma.from_documents(chunks_a, embeddings, persist_directory="/content/ckp02_a")
retriever_a  = db_a.as_retriever(search_kwargs={"k":3})
chain_a      = montar_chain_rag(retriever_a)

# ── ESTRATÉGIA B: SemanticChunker ────────────────────────────────────
# 👉 LACUNA 2: instancie o SemanticChunker com o modelo de embedding
chunks_b    = SemanticChunker(___).split_documents(paginas)
db_b        = Chroma.from_documents(chunks_b, embeddings, persist_directory="/content/ckp02_b")
retriever_b = ___  # retriever com k=3
chain_b     = montar_chain_rag(retriever_b)

# 👉 LACUNA 3: defina 5 perguntas reais do domínio para avaliação
PERGUNTAS = [___, ___, ___, ___, ___]

# 👉 LACUNA 4: monte o dataset RAGAS e avalie as duas estratégias
for nome, chain, retr in [
    ("Recursive-512", chain_a, retriever_a),
    ("Semantic",      chain_b, retriever_b),
]:
    dados = {
        "question": PERGUNTAS,
        "answer":   [chain.invoke(q) for q in PERGUNTAS],
        "contexts": [[d.page_content for d in retr.invoke(q)] for q in ___],
    }
    res = evaluate(Dataset.from_dict(dados), metrics=[faithfulness, answer_relevancy])
    print(f"{nome}: faith={res['faithfulness']:.3f} · rel={res['answer_relevancy']:.3f}")

---

## ✍️ Suas anotações

Registre aqui as observações pedidas no roteiro (qualidade dos resultados, comparações e conclusões do grupo).

## 📚 Referências da aula

- Paper Es, S. et al. — "RAGAS: Automated Evaluation of Retrieval Augmented Generation." EACL, 2024. O paper que define faithfulness e answer_relevancy. arxiv.org/abs/2309.15217
- Docs RAGAS — Documentação oficial: métricas, integração com Ollama, datasets. docs.ragas.io
- Docs LangChain — SemanticChunker e ParentDocumentRetriever. python.langchain.com/docs/how_to/semantic-chunker
- Modelo cross-encoder/ms-marco-MiniLM-L-6-v2 — Modelo de reranking leve (22M params). huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings que sustentam o chunking semântico.

---

**→ Próxima Aula — Aula 08 · 29/Set** — Interfaces com Gradio e Streamlit
  
RAG com URL pública. RunnableWithMessageHistory para memória entre turnos.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*